# Exceptions => Custom Exceptions & Chaining

Custom exceptions give **meaningful names** to your program's own errors. Chaining keeps the original error attached when you raise a new one.

| Tool | Purpose |
|---|---|
| `class MyError(Exception)` | Define a custom exception |
| `super().__init__(message)` | Pass the message to the base class |
| `raise X from Y` | **Explicit** chaining. Stores `Y` in `X.__cause__` |
| Implicit chaining | Raising inside `except` stores the old error in `__context__` |
| `raise X from None` | Hide the chained context |
| `raise` (alone) | Re-raise the current exception |
| `err.add_note("text")` | Attach a note (Python 3.11+). Stored in `__notes__` |
| `ExceptionGroup` / `except*` | Handle several errors at once (Python 3.11+) |

---

## Defining a Custom Exception

```python
class AppError(Exception):
    """Base class for all errors of this app."""

class ValidationError(AppError):
    pass
```

### Naming and Design

* Inherit from `Exception`, **never** from `BaseException`.
* End the name with `Error`.
* Create **one base class** per project or package. Callers can catch everything with `except AppError`.
* Catching a parent class also catches its children.

---

## Adding Data

```python
class InvalidAge(ValueError):
    def __init__(self, age):
        super().__init__(f"Invalid age: {age}")
        self.age = age
```

Extra attributes let handlers react to the details.

---

## Chaining Exceptions

### Implicit: `__context__`

Raising a new exception **inside** an `except` block automatically links the old one.

### Explicit: `raise ... from ...`

```python
try:
    int("abc")
except ValueError as error:
    raise ValidationError("bad number") from error
```

| Attribute | Set by | Meaning |
|---|---|---|
| `__cause__` | `raise X from Y` | The **direct cause** |
| `__context__` | Automatic | The exception being handled when this one was raised |
| `__suppress_context__` | `raise X from None` | Hides the context in the traceback |

The traceback shows both errors, so the original problem is not lost.

---

## Re-raising

A bare `raise` inside `except` re-raises the **same** exception with its original traceback:

```python
except ValueError:
    log("failed")
    raise
```

---

## Notes and Exception Groups (Python 3.11+)

* `error.add_note("context")` adds extra information that is printed after the message.
* `ExceptionGroup("message", [err1, err2])` bundles several errors.
* `except*` handles the matching errors of a group:

```python
try:
    raise ExceptionGroup("batch", [ValueError("a"), TypeError("b")])
except* ValueError as group:
    ...
except* TypeError as group:
    ...
```

---

## EAFP vs LBYL

| Style | Meaning | Example |
|---|---|---|
| **LBYL** | Look Before You Leap: check first | `if key in data: value = data[key]` |
| **EAFP** | Easier to Ask Forgiveness than Permission: try, then handle | `try: value = data[key]` / `except KeyError: ...` |

Python code usually prefers **EAFP**. It avoids race conditions and is often shorter.

## Source

https://docs.python.org/3/tutorial/errors.html#user-defined-exceptions

https://docs.python.org/3/tutorial/errors.html#exception-chaining

https://docs.python.org/3/library/exceptions.html#exception-groups

In [ ]:
# A small exception hierarchy
class AppError(Exception):
    """Base class for all errors of this app."""

class ValidationError(AppError):
    pass

class InvalidAge(ValidationError):
    def __init__(self, age):
        super().__init__(f"Invalid age: {age}")
        self.age = age

def set_age(age):
    if not 0 <= age <= 150:
        raise InvalidAge(age)
    return age

try:
    set_age(200)
except InvalidAge as error:
    print(error, "| age attribute:", error.age)

# Catching the parent class catches the children
try:
    set_age(-5)
except AppError as error:
    print("caught by base class:", type(error).__name__)

print(issubclass(InvalidAge, Exception), issubclass(InvalidAge, ValueError))

# Implicit chaining: __context__
try:
    try:
        1 / 0
    except ZeroDivisionError:
        raise ValidationError("while calculating")
except ValidationError as error:
    print(type(error.__context__).__name__, error.__cause__)

# Explicit chaining: raise ... from ...
def parse_number(text):
    try:
        return int(text)
    except ValueError as error:
        raise ValidationError(f"bad number: {text!r}") from error

try:
    parse_number("abc")
except ValidationError as error:
    print(type(error.__cause__).__name__, "->", error.__cause__)

# raise ... from None hides the context
try:
    try:
        {}["missing"]
    except KeyError:
        raise ValidationError("clean message") from None
except ValidationError as error:
    print(error.__cause__, error.__suppress_context__)

# Re-raising with a bare raise keeps the original exception
try:
    try:
        int("x")
    except ValueError:
        print("logging the failure")
        raise
except ValueError as error:
    print("re-raised:", type(error).__name__)

# Notes (Python 3.11+)
error = ValueError("bad input")
error.add_note("while reading row 12")
print(error.__notes__)

# Exception groups and except* (Python 3.11+)
try:
    raise ExceptionGroup("batch", [ValueError("a"), TypeError("b"), ValueError("c")])
except* ValueError as group:
    print("ValueErrors:", [str(e) for e in group.exceptions])
except* TypeError as group:
    print("TypeErrors:", [str(e) for e in group.exceptions])

# EAFP vs LBYL
data = {"a": 1}
if "b" in data:                                   # LBYL
    value = data["b"]
else:
    value = None

try:                                              # EAFP
    value = data["b"]
except KeyError:
    value = None
print(value)